In [ ]:
import librosa
import numpy as np

def calculate_csig_manual(clean, denoised, sr):
    S_clean = librosa.stft(clean)
    S_den = librosa.stft(denoised)
    power_clean = np.abs(S_clean)**2
    power_den = np.abs(S_den)**2
    correlation = np.corrcoef(power_clean.flatten(), power_den.flatten())[0, 1]
    csig = 1 + 4 * max(0, correlation)
    return min(5.0, max(1.0, csig))

def calculate_cbak_manual(clean, denoised, sr):
    noise = clean - denoised
    noise_energy = np.sum(noise**2)
    signal_energy = np.sum(clean**2)
    snr = 10 * np.log10((signal_energy + 1e-10) / (noise_energy + 1e-10))
    cbak = min(5.0, max(1.0, 1 + snr / 5))
    return cbak

def calculate_covl_manual(csig, cbak):
    covl = 0.114 * csig + 0.093 * cbak + 0.505 * 3.0 + 1.718
    return min(5.0, max(1.0, covl))

def calculate_all_perceptual_metrics(ref_path, deg_path, sr=16000):

    ref, _ = librosa.load(ref_path, sr=sr)
    deg, _ = librosa.load(deg_path, sr=sr)
    
    min_len = min(len(ref), len(deg))
    ref = ref[:min_len]
    deg = deg[:min_len]
    
    csig = calculate_csig_manual(ref, deg, sr)
    cbak = calculate_cbak_manual(ref, deg, sr)
    covl = calculate_covl_manual(csig, cbak)
    
    return {
        'CSIG': csig,
        'CBAK': cbak,
        'COVL': covl
    }

if __name__ == "__main__":
    ref_path = r"\myproject\data\clean_speech.wav"
    deg_path = r"\myproject\output\cleaned_speech.wav"
    
    metrics = calculate_all_perceptual_metrics(ref_path, deg_path)
    
    print("ПЕРЦЕПТУАЛЬНЫЕ МЕТРИКИ")
    
    for name, value in metrics.items():
        if value is not None:
            print(f"{name}: {value:.3f}")

    print("ИНТЕРПРЕТАЦИЯ")
    
    print(f"CSIG (сохранность речи): {metrics['CSIG']:.1f} / 5.0")
    print(f"CBAK (подавление шума): {metrics['CBAK']:.1f} / 5.0")
    print(f"COVL (общее качество): {metrics['COVL']:.1f} / 5.0")

ПЕРЦЕПТУАЛЬНЫЕ МЕТРИКИ
CSIG: 1.015
CBAK: 1.000
COVL: 3.442
ИНТЕРПРЕТАЦИЯ
CSIG (сохранность речи): 1.0 / 5.0
CBAK (подавление шума): 1.0 / 5.0
COVL (общее качество): 3.4 / 5.0
